In [24]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

file_path = list(uploaded.keys())[0] if 'uploaded' in locals() else '1918-2017election_results_by_pcon.xlsx'
print(f"Using {file_path}")

xls = pd.ExcelFile(file_path)

sheets_to_process = ['1970', '1974F', '1974O', '1979', '1983', '1987', '1992', '1997', '2001', '2005', '2010', '2015', '2017']

year_map = {
    '1970': 1970, '1974F': 1974, '1974O': 1974, '1979': 1979, '1983': 1983,
    '1987': 1987, '1992': 1992, '1997': 1997, '2001': 2001, '2005': 2005,
    '2010': 2010, '2015': 2015, '2017': 2017
}

date_map = {
    '1970': '1970-06-18', '1974F': '1974-02-28', '1974O': '1974-10-10',
    '1979': '1979-05-03', '1983': '1983-06-09', '1987': '1987-06-11',
    '1992': '1992-04-09', '1997': '1997-05-01', '2001': '2001-06-07',
    '2005': '2005-05-05', '2010': '2010-05-06', '2015': '2015-05-07',
    '2017': '2017-06-08'
}

party_map = {
    'Conservative': 'Conservative',
    'Conservative Party': 'Conservative',
    'Labour': 'Labour',
    'Labour ': 'Labour',
    'Liberal': 'Liberal Democrats',
    'Liberal Democrat': 'Liberal Democrats',
    'Liberal Democrats': 'Liberal Democrats',
    'Lib Dem': 'Liberal Democrats',
    'Alliance': 'Liberal Democrats',
    'PC/SNP': 'SNP/Plaid Cymru',
    'SNP/Plaid Cymru': 'SNP/Plaid Cymru',
    'Plaid Cymru': 'Plaid Cymru',
    'SNP': 'Scottish National Party',
    'Green': 'Green Party',
    'UKIP': 'UKIP',
    'DUP': 'DUP',
    'Sinn Fein': 'Sinn Fein',
    'SDLP': 'SDLP',
    'UUP': 'UUP',
    'OUP': 'UUP',
    'SF': 'Sinn Fein',
    'Other': 'Other',
    'Speaker': 'Other',
    'Independent': 'Other'
}

all_dfs = []

for sheet in sheets_to_process:
    print(f"\n=== Processing {sheet} ===")
    df = pd.read_excel(xls, sheet_name=sheet, skiprows=2, header=0)

    party_cols = [col for col in df.columns if str(col) not in [f'Unnamed: {i}' for i in range(50)] and pd.notna(col)]

    if len(party_cols) < 3:
        continue

    total_candidates = [col for col in df.columns if 'total' in str(col).lower() and 'votes' in str(col).lower()]
    total_col = total_candidates[0] if total_candidates else df.columns[-2]

    turnout_col = df.columns[-1] if 'turnout' in str(df.columns[-1]).lower() else None

    const_col = df.columns[2] if pd.notna(df.columns[2]) else df.columns[1]

    id_vars = [col for col in df.columns if col not in party_cols]
    if total_col in party_cols: party_cols = [p for p in party_cols if p != total_col]; id_vars.append(total_col)
    if turnout_col and turnout_col in party_cols: party_cols = [p for p in party_cols if p != turnout_col]

    df_long = pd.melt(df, id_vars=id_vars, value_vars=party_cols, var_name='party', value_name='votes')

    df_long['party'] = df_long['party'].str.strip()
    df_long['party'] = df_long['party'].map(party_map).fillna(df_long['party'])

    df_long['votes'] = pd.to_numeric(df_long['votes'], errors='coerce')
    df_long = df_long.dropna(subset=['votes'])

    df_long['Constituency'] = df_long[const_col]
    df_long['election_year'] = year_map[sheet]
    df_long['election_date'] = date_map[sheet]
    df_long['country'] = 'United Kingdom'
    df_long['election_id'] = f'UK_{year_map[sheet]}{"_Feb" if sheet=="1974F" else "_Oct" if sheet=="1974O" else ""}'

    if total_col:
        df_long['vote_share'] = df_long['votes'] / df_long[total_col]

    if turnout_col:
        df_long['turnout'] = pd.to_numeric(df_long[turnout_col].astype(str).str.replace('%', ''), errors='coerce') / 100

    all_dfs.append(df_long)
    print(f"Success: {len(df_long)} records\n")

uk_long = pd.concat(all_dfs, ignore_index=True)

uk_long['seats'] = (uk_long.groupby(['election_year', 'Constituency'])['votes'].transform('max') == uk_long['votes']).astype(int)

uk_long.to_csv('uk_elections_constituency_1970_2017.csv', index=False)

agg_dict = {'votes': 'sum', 'seats': 'sum'}
if 'turnout' in uk_long.columns:
    agg_dict['turnout'] = 'mean'

national = uk_long.groupby(['country', 'election_date', 'election_year', 'election_id', 'party']).agg(agg_dict).reset_index()

national['vote_share'] = national.groupby('election_year')['votes'].transform(lambda x: x / x.sum())
national['seat_share'] = national.groupby('election_year')['seats'].transform(lambda x: x / x.sum())
national['vote_seat_ratio'] = national['seat_share'] / national['vote_share']

national = national.sort_values('election_date').reset_index(drop=True)
national['election_date'] = pd.to_datetime(national['election_date'])

unique_elections = national[['election_date', 'election_year', 'election_id']].drop_duplicates().sort_values('election_date').reset_index(drop=True)

unique_elections['years_since_last_election'] = unique_elections['election_date'].diff().dt.days / 365.25

unique_elections = unique_elections.set_index('election_date')

# Count number of elections (rows) in rolling 10-year window
unique_elections['elections_in_last_10_years'] = unique_elections.index.to_series().rolling(window='3650D', min_periods=1).count() - 1

unique_elections = unique_elections.reset_index()

# Merge back
national = national.merge(unique_elections[['election_date', 'years_since_last_election', 'elections_in_last_10_years']],
                          on='election_date', how='left')
# ======================================

national.to_csv('uk_elections_national_1970_2017.csv', index=False)

print("SUCCESS!")
files.download('uk_elections_constituency_1970_2017.csv')
files.download('uk_elections_national_1970_2017.csv')

Saving 1918-2017election_results_by_pcon.xlsx to 1918-2017election_results_by_pcon (16).xlsx
Using 1918-2017election_results_by_pcon (16).xlsx

=== Processing 1970 ===
Success: 1817 records


=== Processing 1974F ===
Success: 2099 records


=== Processing 1974O ===
Success: 2202 records


=== Processing 1979 ===
Success: 2381 records


=== Processing 1983 ===
Success: 3064 records


=== Processing 1987 ===
Success: 2932 records


=== Processing 1992 ===
Success: 3230 records


=== Processing 1997 ===
Success: 3984 records


=== Processing 2001 ===
Success: 3366 records


=== Processing 2005 ===
Success: 3261 records


=== Processing 2010 ===
Success: 3524 records


=== Processing 2015 ===
Success: 3682 records


=== Processing 2017 ===
Success: 3202 records

SUCCESS! Everything is complete.
Download your clean UK datasets:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
import pandas as pd
from google.colab import files

# Load existing national data (1970–2017)
existing_national = pd.read_csv('uk_elections_national_1970_2017.csv')

# Load 2019 constituency breakdown
df_2019 = pd.read_csv('general_election_vote_breakdown_2019.csv')

# Convert ranked results to long format
rows = []
for i in range(1, 13):
    party_col = '1st_place_party' if i == 1 else f'{i}th_place_party'
    votes_col = '1st_place_votes' if i == 1 else f'{i}th_place_votes'

    if party_col in df_2019.columns and votes_col in df_2019.columns:
        temp = df_2019[[party_col, votes_col]].dropna().copy()
        temp.rename(columns={party_col: 'party', votes_col: 'votes'}, inplace=True)
        temp['votes'] = pd.to_numeric(temp['votes'], errors='coerce')
        rows.append(temp)

long_2019 = pd.concat(rows, ignore_index=True)

# Clean and standardize party names
party_cleanup = {
    'Conservative': 'Conservative',
    'Labour': 'Labour',
    'Liberal Democrat': 'Liberal Democrats',
    'Liberal Democrats': 'Liberal Democrats',
    'Scottish National Party': 'Scottish National Party',
    'Scottish National Party (SNP)': 'Scottish National Party',
    'Plaid Cymru': 'Plaid Cymru',
    'Green Party': 'Green Party',
    'The Brexit Party': 'Brexit Party',
    'UK Independence Party (UKIP)': 'UKIP',
    'Democratic Unionist Party': 'DUP',
    'Sinn Féin': 'Sinn Fein',
    'Social Democratic & Labour Party': 'SDLP',
    'Ulster Unionist Party': 'UUP',
    'Alliance': 'Alliance',
    'Speaker': 'Other'
}

long_2019['party'] = long_2019['party'].str.strip()
long_2019['party'] = long_2019['party'].map(party_cleanup).fillna(long_2019['party'])

#  Aggregate votes nationally
national_2019_votes = long_2019.groupby('party')['votes'].sum().reset_index()

df_2019['winner_party'] = df_2019['1st_place_party'].str.strip()
df_2019['winner_party'] = df_2019['winner_party'].map(party_cleanup).fillna(df_2019['winner_party'])
seats_2019 = df_2019['winner_party'].value_counts().reset_index()
seats_2019.columns = ['party', 'seats']

# Merge votes + seats
national_2019 = national_2019_votes.merge(seats_2019, on='party', how='left')
national_2019['seats'] = national_2019['seats'].fillna(0).astype(int)

#Calculate shares and ratio
total_votes = national_2019['votes'].sum()
total_seats = 650

national_2019['vote_share'] = national_2019['votes'] / total_votes
national_2019['seat_share'] = national_2019['seats'] / total_seats
national_2019['vote_seat_ratio'] = national_2019['seat_share'] / national_2019['vote_share']

national_turnout = (df_2019['total_votes'].sum() / df_2019['registered_voters'].sum()) * 100

# Add metadata
national_2019['country'] = 'United Kingdom'
national_2019['election_date'] = '2019-12-12'
national_2019['election_year'] = 2019
national_2019['election_id'] = 'UK_2019'
national_2019['turnout'] = national_turnout

# Combine with existing data
updated_national = pd.concat([
    existing_national[existing_national['election_year'] != 2019],
    national_2019
], ignore_index=True)

updated_national = updated_national.sort_values('election_date').reset_index(drop=True)
updated_national['election_date'] = pd.to_datetime(updated_national['election_date'])

# Calculate frequency metrics
unique_elections = (updated_national[['election_date', 'election_year', 'election_id']]
                    .drop_duplicates()
                    .sort_values('election_date')
                    .reset_index(drop=True))

unique_elections['years_since_last_election'] = unique_elections['election_date'].diff().dt.days / 365.25

unique_elections = unique_elections.set_index('election_date')
unique_elections['elections_in_last_10_years'] = (
    unique_elections.index.to_series()
    .rolling(window='3650D', min_periods=1)
    .count() - 1
)
unique_elections = unique_elections.reset_index()

# Merge back
updated_national = updated_national.merge(
    unique_elections[['election_date', 'years_since_last_election', 'elections_in_last_10_years']],
    on='election_date',
    how='left'
)

# Final save
updated_national.to_csv('uk_elections_national_1970_2019_clean.csv', index=False)

print("SUCCESS! Clean merged file created: uk_elections_national_1970_2019_clean.csv")
print(f"Total elections: {updated_national['election_year'].nunique()} (1970 to 2019)")
print(f"2019 parties: {len(national_2019)} | Major example: Conservative {national_2019[national_2019['party']=='Conservative']['seats'].values[0]} seats")

files.download('uk_elections_national_1970_2019_clean.csv')

SUCCESS! Clean merged file created: uk_elections_national_1970_2019_clean.csv
Total elections: 13 (1970 to 2019)
2019 parties: 67 | Major example: Conservative 365 seats


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
import pandas as pd
from google.colab import files


# Load existing full UK data
existing_full = pd.read_csv('uk_elections_national_1970_2019_clean.csv')

# Load the new 2024 constituency file
df_2024 = pd.read_csv('general_election_vote_breakdown.csv')  # your new file

rows = []
for i in range(1, 14):
    party_col = '1st_place_party' if i == 1 else f'{i}th_place_party'
    votes_col = '1st_place_votes' if i == 1 else f'{i}th_place_votes'

    if party_col in df_2024.columns and votes_col in df_2024.columns:
        temp = df_2024[[party_col, votes_col]].dropna().copy()
        temp.rename(columns={party_col: 'party', votes_col: 'votes'}, inplace=True)
        temp['votes'] = pd.to_numeric(temp['votes'], errors='coerce')
        rows.append(temp)

long_2024 = pd.concat(rows, ignore_index=True)

party_cleanup_2024 = {
    'Conservative': 'Conservative',
    'Labour': 'Labour',
    'Liberal Democrat': 'Liberal Democrats',
    'Liberal Democrats': 'Liberal Democrats',
    'Scottish National Party': 'Scottish National Party',
    'Plaid Cymru': 'Plaid Cymru',
    'Green Party': 'Green Party',
    'Reform UK': 'Reform UK',
    'Independent': 'Other',
    'Workers Party of Britain': 'Other',
    'Democratic Unionist Party': 'DUP',
    'Sinn Féin': 'Sinn Fein',
    'Social Democratic & Labour Party': 'SDLP',
    'Ulster Unionist Party': 'UUP',
    'Alliance': 'Alliance',
    'Alba Party': 'Other',
    'Trade Unionist and Socialist Coalition': 'Other',
    'Speaker': 'Other'
}

long_2024['party'] = long_2024['party'].str.strip()
long_2024['party'] = long_2024['party'].map(party_cleanup_2024).fillna(long_2024['party'])

# National votes
national_2024_votes = long_2024.groupby('party')['votes'].sum().reset_index()

# Seats
df_2024['winner_party'] = df_2024['1st_place_party'].str.strip()
df_2024['winner_party'] = df_2024['winner_party'].map(party_cleanup_2024).fillna(df_2024['winner_party'])
seats_2024 = df_2024['winner_party'].value_counts().reset_index()
seats_2024.columns = ['party', 'seats']

# Merge
national_2024 = national_2024_votes.merge(seats_2024, on='party', how='left')
national_2024['seats'] = national_2024['seats'].fillna(0).astype(int)

#Shares and ratio
total_votes_2024 = national_2024['votes'].sum()
total_seats_2024 = 650

national_2024['vote_share'] = national_2024['votes'] / total_votes_2024
national_2024['seat_share'] = national_2024['seats'] / total_seats_2024
national_2024['vote_seat_ratio'] = national_2024['seat_share'] / national_2024['vote_share']

national_turnout_2024 = (df_2024['total_votes'].sum() / df_2024['registered_voters'].sum()) * 100

# Metadata
national_2024['country'] = 'United Kingdom'
national_2024['election_date'] = '2024-07-04'  # Official date
national_2024['election_year'] = 2024
national_2024['election_id'] = 'UK_2024'
national_2024['turnout'] = national_turnout_2024

# Append to full dataset
final_national = pd.concat([
    existing_full[existing_full['election_year'] != 2024],
    national_2024
], ignore_index=True)

final_national = final_national.sort_values('election_date').reset_index(drop=True)
final_national['election_date'] = pd.to_datetime(final_national['election_date'])

unique_elections = (final_national[['election_date', 'election_year', 'election_id']]
                    .drop_duplicates()
                    .sort_values('election_date')
                    .reset_index(drop=True))

unique_elections['years_since_last_election'] = unique_elections['election_date'].diff().dt.days / 365.25

unique_elections = unique_elections.set_index('election_date')
unique_elections['elections_in_last_10_years'] = (
    unique_elections.index.to_series()
    .rolling(window='3650D', min_periods=1)
    .count() - 1
)
unique_elections = unique_elections.reset_index()

final_national = final_national.merge(
    unique_elections[['election_date', 'years_since_last_election', 'elections_in_last_10_years']],
    on='election_date',
    how='left'
)

# Save final complete dataset
final_national.to_csv('uk_elections_national_1970_2024.csv', index=False)

print("SUCCESS! UK data now complete: 1970 to 2024")
print(f"Total elections: {final_national['election_year'].nunique()}")
print("2024 highlights:")
print(final_national[final_national['election_year'] == 2024][['party', 'votes', 'seats', 'vote_share', 'seat_share', 'vote_seat_ratio']].round(4))

# Download
files.download('uk_elections_national_1970_2024.csv')

SUCCESS! UK data now complete: 1970 to 2024
Total elections: 14
2024 highlights:
                                  party     votes  seats  vote_share  \
203            Scottish Socialist Party    1007.0      0      0.0001   
204                                SDLP   50516.0      2      0.0030   
205                       Shared Ground     664.0      0      0.0000   
206             Scottish National Party  142939.0      9      0.0085   
207          Scottish Libertarian Party     536.0      0      0.0000   
..                                  ...       ...    ...         ...   
289          English Constitution Party    1563.0      0      0.0001   
290                                 DUP   84367.0      5      0.0050   
291  Cross-Community Labour Alternative     624.0      0      0.0000   
292         Independent Alliance (Kent)     926.0      0      0.0001   
293              Yoruba Party in the UK     261.0      0      0.0000   

     seat_share  vote_seat_ratio  
203      0.0000    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
import pandas as pd
from google.colab import files
df = pd.read_csv('uk_elections_national_1970_2024.csv')

print("Original columns:", df.columns.tolist())

frequency_cols = [
    'years_since_last_election', 'years_since_last_election_x', 'years_since_last_election_y',
    'years_since_last_election_new',
    'elections_in_last_10_years', 'elections_in_last_10_years_x', 'elections_in_last_10_years_y',
    'elections_in_last_10_years_new'
]

df = df.drop(columns=[col for col in frequency_cols if col in df.columns])

df = df.sort_values('election_date').reset_index(drop=True)
df['election_date'] = pd.to_datetime(df['election_date'])

unique_elections = df[['election_date', 'election_year', 'election_id']].drop_duplicates().sort_values('election_date').reset_index(drop=True)

# Years since last election
unique_elections['years_since_last_election'] = unique_elections['election_date'].diff().dt.days / 365.25

# Elections in last 10 years
unique_elections = unique_elections.set_index('election_date')
unique_elections['elections_in_last_10_years'] = (
    unique_elections.index.to_series()
    .rolling(window='3650D', min_periods=1)
    .count() - 1
)
unique_elections = unique_elections.reset_index()

df = df.merge(
    unique_elections[['election_date', 'years_since_last_election', 'elections_in_last_10_years']],
    on='election_date',
    how='left'
)

final_columns = [
    'country', 'election_date', 'election_year', 'election_id', 'party',
    'votes', 'seats', 'vote_share', 'seat_share', 'vote_seat_ratio',
    'turnout', 'years_since_last_election', 'elections_in_last_10_years'
]

df = df[[col for col in final_columns if col in df.columns]]

df.to_csv('uk_elections_national_1970_2024_final.csv', index=False)

print("\nSUCCESS! Fully cleaned file created:")
print("→ uk_elections_national_1970_2024_final.csv")
print("\nFinal columns:", df.columns.tolist())
print(f"Total rows: {len(df)} | Total elections: {df['election_year'].nunique()} (1970–2024)")

# Download the clean file
files.download('uk_elections_national_1970_2024_final.csv')


Original columns: ['country', 'election_date', 'election_year', 'election_id', 'party', 'votes', 'seats', 'vote_share', 'seat_share', 'vote_seat_ratio', 'years_since_last_election_x', 'elections_in_last_10_years_x', 'turnout', 'years_since_last_election_y', 'elections_in_last_10_years_y', 'years_since_last_election', 'elections_in_last_10_years']

SUCCESS! Fully cleaned file created:
→ uk_elections_national_1970_2024_final.csv

Final columns: ['country', 'election_date', 'election_year', 'election_id', 'party', 'votes', 'seats', 'vote_share', 'seat_share', 'vote_seat_ratio', 'turnout', 'years_since_last_election', 'elections_in_last_10_years']
Total rows: 294 | Total elections: 14 (1970–2024)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>